In [34]:
# import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import matplotlib.colors as mcolors
import spacy
from sentence_transformers import SentenceTransformer # needed to do pip uninstall torchcodec -y to make this import
from sklearn.metrics.pairwise import cosine_similarity
import jellyfish
import itertools

In [5]:
# load encoder
encoder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14762.15it/s]


In [6]:
# load data
data = pd.read_csv("/work/verbal_fluency/curvefit/clustered_data_w_condition.csv", encoding="utf-8")

We want to set up a script that does the following:

For each file

    Identify the clusters

        Compute within cluster semantic similarity

            Using embeddings

        Compute within cluster phonemic similarity

            Using jellyfish

In [23]:
cluster_subset.head()

,ID,Filename,Utterance start time,Estimated Lexical Match,Certainty,Target Letter,Audio Quality Comment,Cluster Status,Cluster ID,curve_c,curve_m,Condition
68,0005_FZU,VF-003-20170310114351.wav,50,NaN,NaN,l,low,0,3,8.367628,0.043771,1
69,0005_FZU,VF-003-20170310114351.wav,57,lykke,1.0,l,low,1,3,8.367628,0.043771,1


In [26]:
# extract subset list
subset_list = data["Filename"].unique()

temp_data = []

# looping through the subset list
for j in range(len(subset_list)):

    # constructing a subset with only the first task
    subset = data[data["Filename"] == subset_list[j]]

    # identify number of clusters in that individual file
    n_clusters = max(subset['Cluster ID'])

    # computes word sim
    for i in range(1, n_clusters + 1):

        cluster_subset = subset[subset['Cluster ID'] == i]
        
        if len(cluster_subset) > 0:

            words = cluster_subset['Estimated Lexical Match'].dropna().astype(str).tolist()
            
            if len(words) > 1:

                embeddings = encoder.encode(words)

                sim_matrix = cosine_similarity(embeddings)

                n = len(sim_matrix)

                #print(n)

                avg_sim = np.mean(sim_matrix[np.tril_indices(n)])

                #print(avg_sim)

                cluster_subset['avg_sim'] = avg_sim
            
                temp_data.append(cluster_subset)


In [30]:
data_sim = pd.concat(temp_data, axis=0, ignore_index=True)
data_sim.head()

,ID,Filename,Utterance start time,Estimated Lexical Match,Certainty,Target Letter,Audio Quality Comment,Cluster Status,Cluster ID,curve_c,curve_m,Condition,avg_sim
0,0005_FZU,VF-001-20170310101810.wav,5,solvogn,3.0,s,low,0,1,13.21521,0.030835,3,0.798500
1,0005_FZU,VF-001-20170310101810.wav,7,NaN,1.0,s,low,1,1,13.21521,0.030835,3,0.798500
2,0005_FZU,VF-001-20170310101810.wav,10,snaps,3.0,s,low,1,1,13.21521,0.030835,3,0.798500
3,0005_FZU,VF-001-20170310101810.wav,15,styrevogn,1.0,s,low,0,2,13.21521,0.030835,3,0.879537
4,0005_FZU,VF-001-20170310101810.wav,18,skrue,2.0,s,low,1,2,13.21521,0.030835,3,0.879537


In [49]:
# making a similar one for phonemic similarity

# extract subset list
subset_list = data_sim["Filename"].unique()

temp_data = []

# looping through the subset list
for j in range(len(subset_list)):

    # constructing a subset with only the first task
    subset = data_sim[data_sim["Filename"] == subset_list[j]]

    # identify number of clusters in that individual file
    n_clusters = max(subset['Cluster ID'])

    # computes word sim
    for i in range(1, n_clusters + 1):

        cluster_subset = subset[subset['Cluster ID'] == i]
        
        if len(cluster_subset) > 0:

            words = cluster_subset['Estimated Lexical Match'].dropna().astype(str).tolist()
            
            if len(words) > 1:

                phonetic_string = []

                for w in words:

                    phonetic_string.append(jellyfish.metaphone(w))

                #print(phonetic_string)

                word_pairs = list(itertools.combinations(phonetic_string, 2))

                cluster_sim_list = []

                for pair in word_pairs:

                    str_1 = pair[0]

                    str_2 = pair[1]

                    pair_sim = jellyfish.jaro_winkler_similarity(str_1, str_2)

                    cluster_sim_list.append(pair_sim)
                
                avg_phon_sim = np.mean(cluster_sim_list)

                cluster_subset['avg_phon_sim'] = avg_phon_sim
            
                temp_data.append(cluster_subset)

In [50]:
data_phon_sim = pd.concat(temp_data, axis=0, ignore_index=True)
data_phon_sim.head()

,ID,Filename,Utterance start time,Estimated Lexical Match,Certainty,Target Letter,Audio Quality Comment,Cluster Status,Cluster ID,curve_c,curve_m,Condition,avg_sim,avg_phon_sim
0,0005_FZU,VF-001-20170310101810.wav,5,solvogn,3.0,s,low,0,1,13.21521,0.030835,3,0.798500,0.527778
1,0005_FZU,VF-001-20170310101810.wav,7,NaN,1.0,s,low,1,1,13.21521,0.030835,3,0.798500,0.527778
2,0005_FZU,VF-001-20170310101810.wav,10,snaps,3.0,s,low,1,1,13.21521,0.030835,3,0.798500,0.527778
3,0005_FZU,VF-001-20170310101810.wav,15,styrevogn,1.0,s,low,0,2,13.21521,0.030835,3,0.879537,0.750000
4,0005_FZU,VF-001-20170310101810.wav,18,skrue,2.0,s,low,1,2,13.21521,0.030835,3,0.879537,0.750000


In [56]:
data_phon_sim.to_csv("/work/verbal_fluency/curvefit/clustered_data_w_linguistics.csv", index=False)